In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models

# Verify GPU is working
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

In [ ]:
import os
import numpy as np
import cv2
from sklearn.model_selection import train_test_split
import glob

IMG_HEIGHT = 256
IMG_WIDTH = 256

print("Hunting for the exact file paths and ignoring extensions...")

# 1. Use glob to find the actual images, regardless of extension
img_paths = glob.glob('/kaggle/input/**/img/*.*', recursive=True)

# Filter to only keep paths inside the 'PNG' dataset structure (ignores the JSON dataset metadata)
img_paths = [p for p in img_paths if 'PNG' in p or 'png' in p.lower()]

if not img_paths:
    raise RuntimeError("CRITICAL: Still can't find the files. Ensure the dataset is fully attached.")

# Deduce exact directories dynamically
img_dir = os.path.dirname(img_paths[0])
parent_dir = os.path.dirname(img_dir)
mask_dir = os.path.join(parent_dir, 'masks_machine')

print(f"SUCCESS: Image Directory -> {img_dir}")
print(f"SUCCESS: Mask Directory -> {mask_dir}")

# 2. Robust Data Loading Pipeline
def load_data(img_dir, mask_dir):
    images = []
    masks = []
    
    img_names = sorted(os.listdir(img_dir))
    print(f"Discovered {len(img_names)} image files. Beginning processing...")
    
    for img_name in img_names:
        # Extract the base name (e.g., '102' from '102.jpg')
        base_name = os.path.splitext(img_name)[0]
        
        img_path = os.path.join(img_dir, img_name)
        
        # The mask is almost always a PNG, even if the radiograph is a JPG
        mask_path = os.path.join(mask_dir, f"{base_name}.png")
        
        if not os.path.exists(mask_path):
            continue
            
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        
        if img is None or mask is None:
            continue
            
        img = cv2.resize(img, (IMG_HEIGHT, IMG_WIDTH))
        img = img / 255.0 
        
        mask = cv2.resize(mask, (IMG_HEIGHT, IMG_WIDTH))
        mask = np.where(mask > 0, 1.0, 0.0) 
        
        images.append(img)
        masks.append(mask)
        
    X = np.array(images).reshape(-1, IMG_HEIGHT, IMG_WIDTH, 1)
    y = np.array(masks).reshape(-1, IMG_HEIGHT, IMG_WIDTH, 1)
    
    return X, y

# 3. Execute and Split
X, y = load_data(img_dir, mask_dir)
print(f"\nFinal Data Arrays -> X shape: {X.shape}, y shape: {y.shape}")

if len(X) > 0:
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
    print("\n✅ DATA PIPELINE COMPLETE: You are ready to build the U-Net!")
else:
    print("\n❌ ERROR: Still failing to load arrays.")

In [ ]:
def build_unet(input_shape=(256, 256, 1)):
    inputs = layers.Input(input_shape)

    # --- ENCODER (Contracting Path) ---
    c1 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    c1 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(c1)
    p1 = layers.MaxPooling2D((2, 2))(c1)

    c2 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(p1)
    c2 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(c2)
    p2 = layers.MaxPooling2D((2, 2))(c2)

    c3 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(p2)
    c3 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(c3)
    p3 = layers.MaxPooling2D((2, 2))(c3)

    c4 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(p3)
    c4 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(c4)
    p4 = layers.MaxPooling2D(pool_size=(2, 2))(c4)

    # --- BOTTLENECK ---
    c5 = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(p4)
    c5 = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(c5)

    # --- DECODER (Expanding Path) ---
    u6 = layers.Conv2DTranspose(256, (2, 2), strides=(2, 2), padding='same')(c5)
    u6 = layers.concatenate([u6, c4]) # SKIP CONNECTION
    c6 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(u6)
    c6 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(c6)

    u7 = layers.Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(c6)
    u7 = layers.concatenate([u7, c3]) # SKIP CONNECTION
    c7 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(u7)
    c7 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(c7)

    u8 = layers.Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(c7)
    u8 = layers.concatenate([u8, c2]) # SKIP CONNECTION
    c8 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(u8)
    c8 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(c8)

    u9 = layers.Conv2DTranspose(32, (2, 2), strides=(2, 2), padding='same')(c8)
    u9 = layers.concatenate([u9, c1], axis=3) # SKIP CONNECTION
    c9 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(u9)
    c9 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(c9)

    # Output layer: 1 channel with sigmoid activation for binary classification
    outputs = layers.Conv2D(1, (1, 1), activation='sigmoid')(c9)

    model = models.Model(inputs=[inputs], outputs=[outputs])
    return model

model = build_unet()
model.summary()

In [ ]:
from tensorflow.keras import backend as K

def dice_coef(y_true, y_pred, smooth=1):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def dice_loss(y_true, y_pred):
    return 1.0 - dice_coef(y_true, y_pred)

model.compile(optimizer='adam', loss=dice_loss, metrics=[dice_coef, 'accuracy'])

In [ ]:
# Callbacks to save the best model and stop early if it stops improving
callbacks = [
    tf.keras.callbacks.ModelCheckpoint('tooth_unet_model.h5', verbose=1, save_best_only=True),
    tf.keras.callbacks.EarlyStopping(patience=5, monitor='val_loss')
]

# Train the model
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    batch_size=16,
    epochs=50,
    callbacks=callbacks
)

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy

# 1. CRITICAL: Rebuild the model to clear the ruined weights
model = build_unet()

# 2. Define the combined BCE-Dice Loss for stability
bce = BinaryCrossentropy()
def bce_dice_loss(y_true, y_pred):
    return bce(y_true, y_pred) + dice_loss(y_true, y_pred)

# 3. Lower the Learning Rate to prevent the optimizer from jumping off a cliff
custom_optimizer = Adam(learning_rate=1e-4)

model.compile(optimizer=custom_optimizer, loss=bce_dice_loss, metrics=[dice_coef, 'accuracy'])
print("Model rebuilt and recompiled with BCE-Dice Loss and lower Learning Rate.")

# 4. Update callbacks (Changing to .keras format to fix the warning you received)
callbacks = [
    tf.keras.callbacks.ModelCheckpoint('tooth_unet_model.keras', verbose=1, save_best_only=True, monitor='val_dice_coef', mode='max'),
    tf.keras.callbacks.EarlyStopping(patience=8, monitor='val_dice_coef', mode='max', verbose=1)
]

# 5. Restart Training
print("Restarting training...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    batch_size=16, 
    epochs=50,
    callbacks=callbacks
)

In [ ]:
import matplotlib.pyplot as plt
import random

# Load the best saved model
best_model = tf.keras.models.load_model('tooth_unet_model.keras', custom_objects={'bce_dice_loss': bce_dice_loss, 'dice_coef': dice_coef})

# Pick a random image from the validation set
idx = random.randint(0, len(X_val) - 1)
test_img = X_val[idx]
ground_truth = y_val[idx]

# Expand dimensions to match batch format for prediction (1, 256, 256, 1)
test_img_input = np.expand_dims(test_img, axis=0)

# Generate prediction and threshold it to make a crisp binary mask
prediction = best_model.predict(test_img_input)[0]
prediction_binary = (prediction > 0.5).astype(np.float32)

# Plotting
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.title("Original X-Ray")
plt.imshow(test_img.squeeze(), cmap='gray')

plt.subplot(1, 3, 2)
plt.title("Ground Truth Mask")
plt.imshow(ground_truth.squeeze(), cmap='gray')

plt.subplot(1, 3, 3)
plt.title("U-Net Prediction")
plt.imshow(prediction_binary.squeeze(), cmap='gray')

plt.show()